In [4]:
import polars as pl

# 1. Define schemas
msg_columns = ["time", "event_type", "order_id", "size", "price", "direction"]
ob_columns = []
for i in range(1, 11): 
    ob_columns.extend([f"Ask_Price_{i}", f"Ask_Size_{i}", f"Bid_Price_{i}", f"Bid_Size_{i}"])

# 2. Load the data (REPLACE WITH YOUR EXACT FILE NAMES!)
msg_file = "data/LOBSTER_SampleFile_AMZN_2012-06-21_10/AMZN_2012-06-21_34200000_57600000_message_10.csv" 
ob_file = "data/LOBSTER_SampleFile_AMZN_2012-06-21_10/AMZN_2012-06-21_34200000_57600000_orderbook_10.csv" 

df_messages = pl.read_csv(msg_file, has_header=False, new_columns=msg_columns)
df_orderbook = pl.read_csv(ob_file, has_header=False, new_columns=ob_columns)

# 3. Glue them together horizontally
df_market = pl.concat([df_messages, df_orderbook], how="horizontal")

# 4. Calculate Spread and Mid-Price in dollars
df_market = df_market.with_columns([
    ((pl.col("Ask_Price_1") - pl.col("Bid_Price_1")) / 10000).alias("Spread_Dollars"),
    (((pl.col("Ask_Price_1") + pl.col("Bid_Price_1")) / 2) / 10000).alias("Mid_Price_Dollars")
])

# 5. View the results
print(df_market.select(["time", "event_type", "Spread_Dollars", "Mid_Price_Dollars"]).head(10))

shape: (10, 4)
┌──────────────┬────────────┬────────────────┬───────────────────┐
│ time         ┆ event_type ┆ Spread_Dollars ┆ Mid_Price_Dollars │
│ ---          ┆ ---        ┆ ---            ┆ ---               │
│ f64          ┆ i64        ┆ f64            ┆ f64               │
╞══════════════╪════════════╪════════════════╪═══════════════════╡
│ 34200.01746  ┆ 5          ┆ 0.77           ┆ 223.565           │
│ 34200.189608 ┆ 1          ┆ 0.14           ┆ 223.88            │
│ 34200.189608 ┆ 1          ┆ 0.14           ┆ 223.88            │
│ 34200.189608 ┆ 1          ┆ 0.14           ┆ 223.88            │
│ 34200.189608 ┆ 1          ┆ 0.14           ┆ 223.88            │
│ 34200.189608 ┆ 1          ┆ 0.14           ┆ 223.88            │
│ 34200.189608 ┆ 1          ┆ 0.14           ┆ 223.88            │
│ 34200.189608 ┆ 1          ┆ 0.14           ┆ 223.88            │
│ 34200.189608 ┆ 1          ┆ 0.14           ┆ 223.88            │
│ 34200.189608 ┆ 1          ┆ 0.14           ┆ 

/tmp/ipykernel_19549/1376694075.py:17: DeprecationWarning: the default behavior of `how='horizontal'` for `concat` is deprecated and will require equal heights in the next breaking release. Use `how='horizontal_extend'` to keep the current behavior.
(Deprecated in version 1.42.1)
  df_market = pl.concat([df_messages, df_orderbook], how="horizontal")
